# Project Proposal
## Skin-Lesion Classification on HAM10000 — Combining Metadata Statistics and Machine Learning

**Course:** Introduction to Data Science
**Instructor:** Quách Đình Hoàng
**Team 07:** Nguyễn Nhật Phát, Bùi Trần Tấn Phát
**Proposal deadline:** 29 March 2026

---

### Table of contents
1. [Problem and dataset](#1)
2. [Research questions](#2)
3. [Data-analysis plan](#3)
4. [Evaluation metrics](#4)
5. [Timeline and task division](#5)
6. [Expected outcomes](#6)
7. [References](#7)

## 1. Problem and dataset  <a id="1"></a>

### 1.1 Motivation

Melanoma is one of the skin cancers whose prognosis depends most strongly on the stage at detection. According to the U.S. National Cancer Institute SEER program, the 5-year relative survival rate for melanoma is **97.6%** when it is diagnosed while still *localized*, but drops to **16.2%** once it reaches *distant* sites [1]. This large clinical gap is what makes automated triage of dermoscopic images worthwhile: even a moderate improvement in early-stage referral can translate into a meaningful reduction in mortality.

Recent work on AI-assisted skin-lesion classification reports sensitivities around **93.6%** and specificities around **70.4%** on curated image sets, confirming that machine-learning systems can act as a useful second reader for dermatologists [2]. From the perspective of *Introduction to Data Science*, the problem is attractive because it naturally combines three course topics on a single real-world dataset: descriptive statistics on tabular metadata, formal hypothesis testing, and supervised learning.

### 1.2 Dataset — HAM10000

We use the **HAM10000** ("Human Against Machine with 10000 training images") dataset, published in *Scientific Data* in 2018 [3] and also distributed through the Harvard Dataverse (DOI `10.7910/DVN/DBW86T`) [4]. The collection contains **10,015 dermatoscopic images** gathered over roughly 20 years at two clinical centres — the Department of Dermatology of the Medical University of Vienna (Austria) and a skin-cancer practice in Queensland (Australia). Ground-truth labels were established through histopathology, clinical follow-up, expert consensus, or in-vivo confocal microscopy, and the resulting heterogeneity is one of the aspects we plan to discuss in the report [3][4].

HAM10000 covers **seven diagnostic classes**: melanocytic nevi (`nv`), melanoma (`mel`), benign keratosis-like lesions (`bkl`), basal-cell carcinoma (`bcc`), actinic keratoses / intraepithelial carcinoma (`akiec`), vascular lesions (`vasc`), and dermatofibroma (`df`). The accompanying metadata file `HAM10000_metadata.csv` contains the variables `lesion_id`, `image_id`, `dx`, `dx_type`, `age`, `sex`, and `localization`.

Beyond the 4 metadata predictors, we plan to extract a further **~50 handcrafted image descriptors** (color statistics, brightness, contrast, asymmetry, border irregularity, LBP and GLCM texture). With 10,015 observations and well over 10 informative variables spanning categorical, discrete and continuous types, the dataset fully satisfies the scale and variety requirements stated in the course guide.

## 2. Research questions  <a id="2"></a>

The project is framed as a **multi-class classification** problem with the diagnostic label `dx` as the response variable, and is structured around three research questions:

> **RQ1.** Are demographic and clinical variables (age, sex, localization, `dx_type`) significantly associated with the lesion type?
>
> **RQ2.** Can a tabular classifier built on metadata plus handcrafted image features achieve acceptable multi-class performance, particularly for melanoma?
>
> **RQ3.** Does combining metadata with image features outperform either feature set alone?

These questions cover both the *statistical* and the *predictive* side of data science, which makes them a natural fit for the Proposal / Milestone / Report workflow defined by the course.

## 3. Data-analysis plan  <a id="3"></a>

### 3.1 Inputs and outputs

- **Response (Y):** `dx` — 7-class categorical diagnosis.
- **Predictors (X):**
  - *From metadata:* `age` (continuous), `sex` (categorical), `dx_type` (categorical), `localization` (15 categorical levels).
  - *From images:* mean / std per RGB channel, HSV statistics, global brightness, contrast, asymmetry index, border irregularity, Local Binary Pattern histogram, and GLCM texture descriptors.

### 3.2 Exploratory data analysis

- Summary statistics, missing-value audit, dtype check.
- Distribution of `age`, `sex`, `localization`, and `dx`.
- Visualisation: histograms, box-plots, count-plots, stacked bar charts, and correlation heat-maps.
- Explicit quantification of class imbalance (we expect `nv` ≈ 67%).

### 3.3 Statistical hypothesis tests

| Test | Variables | Hypothesis |
|---|---|---|
| Chi-square | `sex` vs `dx` | H0: independence between sex and diagnosis |
| Chi-square | `localization` vs `dx` | H0: body site is independent of diagnosis |
| Kruskal–Wallis | `age` across `dx` groups | H0: equal age distribution across diagnoses |
| Dunn post-hoc (Bonferroni) | pairwise `dx` on `age` | localise the Kruskal–Wallis rejection |

We prefer Kruskal–Wallis over ANOVA because the age distribution is visibly non-Gaussian and contains outliers. All tests are run at α = 0.05.

### 3.4 Image-feature extraction

Each image is resized to 224 × 224 and summarised by ~50 handcrafted descriptors grouped into color (RGB / HSV statistics), shape (asymmetry, border irregularity), and texture (LBP histogram, GLCM contrast / dissimilarity / homogeneity / energy / correlation). Features are cached in `results/image_features.csv` so that downstream modelling becomes a cheap tabular task.

### 3.5 Preprocessing

- Median imputation of missing `age`.
- One-hot encoding for `sex`, `dx_type`, `localization`.
- `StandardScaler` on all continuous features.
- Stratified train/test split (80 / 20) on `dx` with a fixed random seed.
- Class-imbalance handling: `class_weight='balanced'` and/or **SMOTE** on the training set only.

### 3.6 Candidate models

| Model | Role |
|---|---|
| Logistic Regression | Linear baseline, easy to interpret |
| K-Nearest Neighbors | Non-parametric reference |
| Random Forest | Robust baseline; native feature importance |
| Support Vector Machine (RBF) | Strong on mid-dimensional feature sets |
| XGBoost | Leading gradient-boosted tree |
| LightGBM | Faster gradient-boosting alternative; **expected main model** |

The best model is chosen by a *composite* criterion (macro-F1 + per-class recall, not pure accuracy). We also plan to run a dedicated **melanoma-vs-rest** experiment combined with threshold tuning, because from a medical point of view a false negative on `mel` is much costlier than a false positive on `nv`.

## 4. Evaluation metrics  <a id="4"></a>

Because the dataset is heavily imbalanced, raw accuracy is a misleading headline number. We commit in advance to reporting:

- **Accuracy** (context only).
- **Precision / Recall / F1** per class.
- **Macro-F1** — our *primary* metric.
- **Weighted-F1** — sanity check aligned with class frequencies.
- **ROC-AUC (one-vs-rest)** — probability-ranking quality.
- **Confusion matrices** — to inspect which classes are most confused (we expect `mel` ↔ `nv`).
- **Melanoma recall** at the default threshold and at tuned operating points.

For model selection we perform **5-fold Stratified Cross-Validation** on the training split before retraining on the full training set and evaluating on the held-out test split.

## 5. Timeline and task division  <a id="5"></a>

### 5.1 Schedule

| Week | Dates | Milestone |
|---|---|---|
| 1 | 22 – 28 Mar | Dataset inspection, EDA scaffolding, proposal writing |
| 2 | 29 Mar – 04 Apr | Statistical tests; handcrafted image-feature extraction |
| 3 | 05 – 11 Apr | Baseline models, 5-fold CV, Milestone deliverable |
| 4 | 12 – 18 Apr | Hyperparameter tuning (LightGBM / XGBoost); feature-set comparison |
| 5 | 19 – 22 Apr | Melanoma-focused improvements, presentation rehearsal |
| 6 | 23 – 26 Apr | Final report, HTML/PDF export, peer assessment |

### 5.2 Task division

| Member | Primary responsibilities |
|---|---|
| **Nguyễn Nhật Phát** | Dataset description, EDA on metadata, statistical tests (Chi-square, Kruskal–Wallis, Dunn post-hoc), visualisations, proposal & report narrative. |
| **Bùi Trần Tấn Phát** | Handcrafted image-feature extraction pipeline, preprocessing, model training, 5-fold CV, hyperparameter tuning, melanoma-focused pipeline, modelling narrative. |
| *Shared* | Code review, insight discussion, slide preparation, Q&A rehearsal, report polishing. |

The split is symmetric in workload and gives each member end-to-end ownership of one of the two analysis tracks (statistical vs. predictive), which matches the peer-assessment rubric.

## 6. Expected outcomes  <a id="6"></a>

We expect three concrete outputs:

1. **Statistically significant associations** between patient metadata and lesion type — in particular between `localization` and `dx`, and between `age` and `dx`.
2. **A tuned gradient-boosted tree (LightGBM/XGBoost)** reaching roughly 0.80 accuracy and 0.94 ROC-AUC on the held-out test split; combining metadata + image features should clearly beat either single feature set.
3. **A melanoma-focused pipeline** (threshold tuning + binary mel-vs-rest gate + two-tier model) that noticeably improves `mel` recall over the default argmax, at a controlled cost in overall macro-F1.

We will explicitly discuss the main limitations: class imbalance, heterogeneous ground-truth (not all labels are histologically verified), the fact that multiple images can share the same `lesion_id`, and the absence of an external validation set.

## 7. References  <a id="7"></a>

[1] National Cancer Institute, *SEER Cancer Stat Facts: Melanoma of the Skin*.

[2] Mevorach, L., *et al.* "A comparison of skin-lesions' diagnoses between AI-based classifiers and dermatologists," 2025.

[3] Tschandl, P., Rosendahl, C., & Kittler, H. "The HAM10000 dataset, a large collection of multi-source dermatoscopic images of common pigmented skin lesions," *Scientific Data*, 5, 180161, 2018. DOI: 10.1038/sdata.2018.161.

[4] Harvard Dataverse, *The HAM10000 dataset*, DOI: 10.7910/DVN/DBW86T.